In [2]:
from google.colab import files

uploaded = files.upload()

Saving suppliers_cleaned.csv to suppliers_cleaned.csv
Saving returns_cleaned.csv to returns_cleaned.csv
Saving products_cleaned.csv to products_cleaned.csv
Saving payments_cleaned.csv to payments_cleaned.csv
Saving orders_cleaned.csv to orders_cleaned.csv
Saving order_items_cleaned.csv to order_items_cleaned.csv
Saving inventory_cleaned.csv to inventory_cleaned.csv
Saving customers_cleaned.csv to customers_cleaned.csv


In [3]:
import os

for file in os.listdir():
    if file.endswith(".csv"):
        print(file)

customers_cleaned.csv
inventory_cleaned.csv
products_cleaned.csv
suppliers_cleaned.csv
returns_cleaned.csv
orders_cleaned.csv
payments_cleaned.csv
order_items_cleaned.csv


In [4]:
import pandas as pd

customers = pd.read_csv("customers_cleaned.csv")
products = pd.read_csv("products_cleaned.csv")
payments = pd.read_csv("payments_cleaned.csv")
order_items = pd.read_csv("order_items_cleaned.csv")
orders = pd.read_csv("orders_cleaned.csv")
returns = pd.read_csv("returns_cleaned.csv")
suppliers = pd.read_csv("suppliers_cleaned.csv")
inventory = pd.read_csv("inventory_cleaned.csv")

orders["order_date"] = pd.to_datetime(orders["order_date"])
customers["registration_date"] = pd.to_datetime(customers["registration_date"])
payments["payment_date"] = pd.to_datetime(payments["payment_date"])
returns["return_date"] = pd.to_datetime(returns["return_date"])

print("✅ All 8 cleaned datasets loaded successfully!")

✅ All 8 cleaned datasets loaded successfully!


KPI 1 — Gross Merchandise Value (GMV).

In [5]:
gmv = orders["final_amount"].sum()

print("Gross Merchandise Value (GMV): ₹", round(gmv, 2))

Gross Merchandise Value (GMV): ₹ 3147385544.58


KPI 2 — Net Revenue

In [6]:
net_revenue = orders.loc[
    orders["status"] == "Delivered",
    "final_amount"
].sum()

print("Net Revenue: ₹", round(net_revenue, 2))

Net Revenue: ₹ 2053858611.96


KPI 3 — Average Order Value (AOV)

In [7]:
delivered_orders = (orders["status"] == "Delivered").sum()

aov = net_revenue / delivered_orders

print("Delivered Orders:", delivered_orders)
print("Average Order Value (AOV): ₹", round(aov, 2))

Delivered Orders: 32499
Average Order Value (AOV): ₹ 63197.59


KPI 4 — Cancellation Rate

In [8]:
cancelled_orders = (orders["status"] == "Cancelled").sum()
total_orders = len(orders)

cancellation_rate = (cancelled_orders / total_orders) * 100

print("Total Orders:", total_orders)
print("Cancelled Orders:", cancelled_orders)
print("Cancellation Rate:", round(cancellation_rate, 2), "%")

Total Orders: 50000
Cancelled Orders: 5996
Cancellation Rate: 11.99 %


KPI 5 — Return Rate

In [9]:
return_records = len(returns)

return_rate = (return_records / delivered_orders) * 100

print("Return Records:", return_records)
print("Delivered Orders:", delivered_orders)
print("Return Rate:", round(return_rate, 2), "%")

Return Records: 10000
Delivered Orders: 32499
Return Rate: 30.77 %


KPI 6 — Customer Lifetime Value (CLV)

In [10]:
clv_by_segment = (
    customers
    .groupby("segment")["total_spent"]
    .mean()
    .sort_values(ascending=False)
)

print("Customer Lifetime Value (CLV) by Segment:")
print(clv_by_segment.round(2))

Customer Lifetime Value (CLV) by Segment:
segment
Premium     532735.00
Regular     361870.72
New         274866.48
Budget      186178.81
Inactive     32249.08
Name: total_spent, dtype: float64


Observation: Premium customers have the highest average lifetime spending at ₹532,735, while Inactive customers have the lowest at ₹32,249.08. Premium CLV is about 1.47× Regular, so it does not meet the 3× benchmark.

KPI 7 — Month-over-Month Growth

In [12]:
monthly_gmv = (
    orders
    .groupby(orders["order_date"].dt.to_period("M"))["final_amount"]
    .sum()
    .sort_index()
)

mom_growth = monthly_gmv.pct_change() * 100

mom_growth = mom_growth.round(2)

print("Month-over-Month GMV Growth:")
print(mom_growth)

Month-over-Month GMV Growth:
order_date
2023-06       NaN
2023-07    370.77
2023-08     -0.62
2023-09      7.29
2023-10     -3.63
2023-11      3.62
2023-12     -0.25
2024-01     -4.00
2024-02     -7.99
2024-03     10.52
2024-04     -1.62
2024-05      3.69
2024-06     -9.25
2024-07      2.66
2024-08      3.85
2024-09     -1.06
2024-10      2.29
2024-11     -5.43
2024-12      5.41
2025-01      3.64
2025-02    -17.26
2025-03     19.61
2025-04     -4.15
2025-05      3.84
2025-06    -23.74
Freq: M, Name: final_amount, dtype: float64


KPI 8 — Top Category Revenue Share

In [13]:
category_revenue = (
    order_items
    .groupby("category")["total_price"]
    .sum()
    .sort_values(ascending=False)
)

top_category = category_revenue.index[0]
top_category_revenue = category_revenue.iloc[0]
total_category_revenue = category_revenue.sum()

top_category_share = (
    top_category_revenue / total_category_revenue
) * 100

print("Top Category:", top_category)
print("Top Category Revenue: ₹", round(top_category_revenue, 2))
print("Top Category Revenue Share:", round(top_category_share, 2), "%")

Top Category: Electronics
Top Category Revenue: ₹ 1642422813.39
Top Category Revenue Share: 57.41 %


KPI 9 — Payment Failure Rate

In [14]:
total_payments = len(payments)
failed_payments = (payments["status"] == "Failed").sum()

payment_failure_rate = (
    failed_payments / total_payments
) * 100

print("Total Payments:", total_payments)
print("Failed Payments:", failed_payments)
print("Payment Failure Rate:", round(payment_failure_rate, 2), "%")

Total Payments: 50000
Failed Payments: 1748
Payment Failure Rate: 3.5 %


KPI 10 — Inventory Fill Rate

In [16]:
total_skus = len(inventory)
in_stock_skus = (inventory["status"] == "In Stock").sum()

inventory_fill_rate = (
    in_stock_skus / total_skus
) * 100

print("Total SKUs:", total_skus)
print("In Stock SKUs:", in_stock_skus)
print("Inventory Fill Rate:", round(inventory_fill_rate, 2), "%")


Total SKUs: 1000
In Stock SKUs: 922
Inventory Fill Rate: 92.2 %


Key management insights
GMV: ₹3.15 billion indicates a large overall business volume.
Net Revenue: ₹2.05 billion came from delivered orders.
Cancellation rate: 11.99%, above the 10% target.
Return rate: 30.77%, significantly above the 8% target and the biggest concern.
Payment failures: 3.50%, above the 2% target.
Inventory fill rate: 92.2%, below the 95% target.
Electronics: contributes 57.41% of category revenue, close to the 60% concentration-risk threshold.
Premium CLV: ₹532,735, but only about 1.47× Regular CLV, rather than the 3× benchmark.

